In [ ]:
!pip install catboost lightgbm xgboost -q

In [ ]:
# ========================================
# IMPORT MODEL KLASIFIKASI - SKLEARN
# ========================================

# --- Linear Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import PassiveAggressiveClassifier

# --- Naive Bayes ---
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB

# --- Support Vector Machine ---
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# --- Decision Tree ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import ExtraTreeClassifier

# --- Ensemble ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# --- Neighbors ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NearestCentroid

# --- Discriminant Analysis ---
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# --- Neural Network ---
from sklearn.neural_network import MLPClassifier

# --- Boosting Lanjutan ---
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Save model
import pickle

# ============================================================
# IMPORT LENGKAP - PREPROCESSING, SPLITTING, TUNING, METRIK
# ============================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')


## Business Understanding

### Background Context
Sebuah perusahaan **penerbit kartu kredit** ingin mengetahui nasabah mana yang berpotensi **gagal bayar (default)** pada bulan berikutnya, berdasarkan riwayat limit kredit, status pembayaran, dan tagihan 6 bulan terakhir.

### Problem Statement
Perusahaan belum memiliki cara sistematis untuk mengidentifikasi nasabah yang berisiko gagal membayar tagihan kartu kredit bulan depan, sehingga proses mitigasi risiko (penagihan dini, penyesuaian limit, dsb) tidak bisa dilakukan secara proaktif.

### Goal
Membangun model *machine learning* yang mampu memprediksi apakah seorang nasabah akan **default (gagal bayar)** pada bulan berikutnya (`default.payment.next.month` = 1) atau tidak (0).

### Stakeholder
Head of Credit Risk Division

### Analytical Approach
Klasifikasi biner (*binary classification*) menggunakan *machine learning*, dievaluasi terutama dengan **F1-score** dan **ROC-AUC** karena target bersifat *imbalanced* (~22% default).

### Deskripsi Kolom (Data Dictionary)
| Kolom | Keterangan |
|---|---|
| ID | ID unik nasabah |
| LIMIT_BAL | Limit kredit (NT dollar) |
| SEX | 1 = pria, 2 = wanita |
| EDUCATION | 1=graduate school, 2=university, 3=high school, 4=others |
| MARRIAGE | 1=menikah, 2=lajang, 3=lainnya |
| AGE | Usia |
| PAY_0, PAY_2..PAY_6 | Status pembayaran bulan-bulan sebelumnya (-1=lunas tepat waktu, 1-9=terlambat n bulan) |
| BILL_AMT1..6 | Jumlah tagihan tiap bulan |
| PAY_AMT1..6 | Jumlah yang dibayarkan tiap bulan |
| default.payment.next.month | Target: 1 = default, 0 = tidak default |


In [ ]:
df = pd.read_csv('UCI_Credit_Card.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

## Data Cleaning
Menangani data duplikat, outlier, dan data yang tidak relevan.

In [ ]:
# Cek duplikat
print("Jumlah baris duplikat:", df.duplicated().sum())

# Kolom ID tidak relevan untuk pemodelan (hanya identifier), akan dibuang nanti dari fitur
print("Jumlah nilai unik ID:", df['ID'].nunique(), "dari", len(df), "baris")

In [ ]:
# Cek missing value
df.isnull().sum()

In [ ]:
# Cek kategori tidak wajar pada EDUCATION dan MARRIAGE (menurut dokumentasi UCI seharusnya EDUCATION 1-4, MARRIAGE 1-3)
print(df['EDUCATION'].value_counts())
print()
print(df['MARRIAGE'].value_counts())

In [ ]:
# Rapikan kategori EDUCATION: 0, 5, 6 dianggap "others" -> gabung ke kategori 4
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# Rapikan kategori MARRIAGE: 0 dianggap "others" -> gabung ke kategori 3
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print(df['EDUCATION'].value_counts())
print()
print(df['MARRIAGE'].value_counts())

## Exploratory Data Analysis (EDA)
Menganalisis hubungan fitur dengan target secara visual.

In [ ]:
# Distribusi target (cek imbalance)
plt.figure(figsize=(5,4))
sns.countplot(x='default.payment.next.month', data=df)
plt.title('Distribusi Target: Default vs Tidak Default')
plt.xlabel('0 = Tidak Default, 1 = Default')
plt.show()

df['default.payment.next.month'].value_counts(normalize=True)

In [ ]:
# Distribusi LIMIT_BAL terhadap target
plt.figure(figsize=(6,4))
sns.boxplot(x='default.payment.next.month', y='LIMIT_BAL', data=df)
plt.title('Limit Kredit vs Status Default')
plt.show()

In [ ]:
# Korelasi antar fitur numerik
plt.figure(figsize=(12,10))
sns.heatmap(df.drop(columns=['ID']).corr(), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Rata-rata status pembayaran (PAY_0) terhadap target
sns.countplot(x='PAY_0', hue='default.payment.next.month', data=df)
plt.title('Status Pembayaran Bulan Terakhir (PAY_0) vs Default')
plt.show()

## Define Feature & Label

In [ ]:
# Feature: semua kolom kecuali ID (identifier) dan target
feature_cols = [c for c in df.columns if c not in ['ID', 'default.payment.next.month']]

X = df[feature_cols]
y = df['default.payment.next.month']

print("Jumlah fitur:", len(feature_cols))
X.head()

## Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=10
)

# train -> untuk melatih model, test -> untuk evaluasi model
print(X_train.shape, X_test.shape)

## Preprocessor Pipeline
Seluruh fitur pada dataset ini bersifat **numerik** (termasuk SEX, EDUCATION, MARRIAGE, dan PAY_x yang berupa kode angka), sehingga preprocessing difokuskan pada imputasi & scaling numerik. Tidak ada kolom bertipe teks/kategorikal string di dataset ini.

In [ ]:
numeric_features = feature_cols  # semua fitur numerik

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ])

## Modeling

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SGD Classifier": SGDClassifier(loss='log_loss'),
    "Ridge Classifier": RidgeClassifier(),
    "Passive Aggressive": PassiveAggressiveClassifier(),
    "Gaussian NB": GaussianNB(),
    "Bernoulli NB": BernoulliNB(),
    "Linear SVC": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Extra Tree (single)": ExtraTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Bagging": BaggingClassifier(),
    "Extra Trees": ExtraTreesClassifier(),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Nearest Centroid": NearestCentroid(),
    "LDA": LinearDiscriminantAnalysis(),
    "MLP Neural Network": MLPClassifier(max_iter=500),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier(verbose=-1),
    "CatBoost": CatBoostClassifier(verbose=0)
}
# Catatan: SVC dan NuSVC sengaja tidak dimasukkan karena dataset besar (30rb baris)
# akan membuat training sangat lambat (kompleksitas O(n^2) - O(n^3)).

def benchmark_models(X_train, X_test, y_train, y_test):
    results = []
    print("Mulai proses benchmarking. Proses ini mungkin memakan waktu...\n")
    for model_name, model in models.items():
        try:
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', model)
            ])
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)

            if hasattr(pipeline.named_steps['classifier'], "predict_proba"):
                y_scores = pipeline.predict_proba(X_test)[:, 1]
            elif hasattr(pipeline.named_steps['classifier'], "decision_function"):
                y_scores = pipeline.decision_function(X_test)
            else:
                y_scores = None

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            roc_auc = roc_auc_score(y_test, y_scores) if y_scores is not None else np.nan

            results.append({'Model': model_name, 'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': roc_auc})
            print(f"\u2705 Selesai: {model_name}")
        except Exception as e:
            print(f"\u274c Gagal: {model_name} (Alasan: {str(e).split('.')[0]})")
            results.append({'Model': model_name, 'Accuracy': np.nan, 'F1-Score': np.nan, 'ROC-AUC': np.nan})

    results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
    return results_df

hasil = benchmark_models(X_train, X_test, y_train, y_test)
hasil

## Hyperparameter Tuning
Tuning dilakukan pada beberapa model kandidat terbaik dari hasil benchmarking di atas.

In [ ]:
tuning_configs = [
    {
        'name': 'CatBoost',
        'model': CatBoostClassifier(verbose=0),
        'params': {
            'model__iterations': [100, 200],
            'model__depth': [4, 6]
        }
    },
    {
        'name': 'LightGBM',
        'model': LGBMClassifier(verbose=-1),
        'params': {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1]
        }
    },
    {
        'name': 'Hist Gradient Boosting',
        'model': HistGradientBoostingClassifier(),
        'params': {
            'model__max_iter': [100, 200],
            'model__learning_rate': [0.05, 0.1]
        }
    },
    {
        'name': 'XGBoost',
        'model': XGBClassifier(eval_metric='logloss'),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5]
        }
    },
    {
        'name': 'Random Forest',
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [None, 10, 20],
            'model__min_samples_split': [2, 5]
        }
    }
]

def tune_top_models(X_train, y_train):
    best_models = {}
    tuning_results = []
    print("Mulai Hyperparameter Tuning (Mode: Cepat/Tidak Greedy)...\n")

    for config in tuning_configs:
        model_name = config['name']
        model = config['model']
        params = config['params']

        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        grid_search = GridSearchCV(
            pipeline,
            param_grid=params,
            cv=3,
            scoring='f1',
            n_jobs=-1
        )

        print(f"\U0001f504 Tuning {model_name}...")
        grid_search.fit(X_train, y_train)

        best_models[model_name] = grid_search.best_estimator_
        tuning_results.append({
            'Model': model_name,
            'Best F1 (CV)': grid_search.best_score_,
            'Best Parameters': grid_search.best_params_
        })

        print(f"\u2705 Selesai {model_name} | Best F1: {grid_search.best_score_:.4f}")
        print(f"   Parameter: {grid_search.best_params_}\n")

    return best_models, pd.DataFrame(tuning_results).sort_values(by='Best F1 (CV)', ascending=False)

model_terbaik_dict, tabel_hasil_tuning = tune_top_models(X_train, y_train)
tabel_hasil_tuning

## Evaluasi Model Terbaik pada Data Test

In [ ]:
nama_model_terbaik = tabel_hasil_tuning.iloc[0]['Model']
best_pipeline = model_terbaik_dict[nama_model_terbaik]

y_pred_test = best_pipeline.predict(X_test)
if hasattr(best_pipeline.named_steps['model'], 'predict_proba'):
    y_scores_test = best_pipeline.predict_proba(X_test)[:, 1]
else:
    y_scores_test = best_pipeline.decision_function(X_test)

print(f"Model terbaik: {nama_model_terbaik}\n")
print(classification_report(y_test, y_pred_test))
print("ROC-AUC:", roc_auc_score(y_test, y_scores_test))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Default','Default'],
            yticklabels=['Tidak Default','Default'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title(f'Confusion Matrix - {nama_model_terbaik}')
plt.show()

## Simpan Model Final
Model final di-*fit ulang* pada seluruh data (`X`, `y`) sebelum disimpan, supaya memanfaatkan seluruh data yang tersedia untuk versi produksi.

In [ ]:
best_model = model_terbaik_dict[nama_model_terbaik]
best_model.fit(X, y)

with open('best_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)

print(f"\u2705 Model berhasil disimpan ke file: best_model.pkl")

# Cara load model kembali:
# with open('best_model.pkl', 'rb') as file:
#     loaded_model = pickle.load(file)
# loaded_model.predict(X_data_baru)

## Limitasi Model
Model hanya valid diterapkan pada data baru yang berada dalam rentang nilai fitur berikut (di luar rentang ini, prediksi model kurang bisa diandalkan / *extrapolation*).

In [ ]:
X.describe().T[['min','max']]

**Catatan limitasi tambahan:**
- Dataset bersifat *imbalanced* (~22% kelas default), sehingga model cenderung lebih baik memprediksi kelas mayoritas (tidak default); perlu dipantau *recall* pada kelas default.
- Model dilatih pada data historis nasabah Taiwan (dataset UCI), sehingga generalisasi ke populasi/negara lain perlu divalidasi ulang.
- Fitur PAY_0..PAY_6, BILL_AMT, PAY_AMT bersifat *time-dependent* (snapshot 6 bulan); performa bisa menurun jika pola perilaku pembayaran nasabah berubah signifikan dari waktu ke waktu (*data drift*).
